<a href="https://colab.research.google.com/github/samcamacho62-dotcom/-ITAI-1371-ML-Labs/blob/main/cleaned_ready_data_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Create a realistic messy enterprise dataset
raw_data = {
    "Emp_ID": ["E101", "E102", "E102", "E103", "E104", "E105", "E106", "E107"],
    "Full Name ": [
        "  john smith ", "SARAH CONNOR", "SARAH CONNOR",
        "mike_tyson", " Angela Camacho ", "david_goggins",
        "Elena Fisher", "Marcus Fenix"
    ],
    "Dept_Region": [
        "IT-Houston", "Cyber-Houston", "Cyber-Houston",
        "AI_Ops-Dallas", "IT-Austin", None,
        "Cyber-Houston", "AI_Ops-Houston"
    ],
    "Salary_USD": [
        "$72,500", "$95,000", "$95,000",
        "N/A", "68000", "$110,200",
        "$-55,000", None
    ],
    "Age": [28, 34, 34, np.nan, 29, 41, 195, np.nan],
    "Hire_Date": [
        "2024-03-15", "05/20/2023", "05/20/2023",
        "2025/01/10", "invalid_date", "2022-11-01",
        "2024-07-22", "2026-02-14"
    ],
    "Performance_Score": [4.5, 4.9, 4.9, np.nan, 4.2, 5.0, 3.8, np.nan]
}

df_raw = pd.DataFrame(raw_data)
df_raw.to_csv("raw_messy_data.csv", index=False)

print("=== BEFORE CLEANING (RAW DATA) ===")
display(df_raw)
print("\nMissing values count:\n", df_raw.isnull().sum())

=== BEFORE CLEANING (RAW DATA) ===


,Emp_ID,Full Name,Dept_Region,Salary_USD,Age,Hire_Date,Performance_Score
0,E101,john smith,IT-Houston,"$72,500",28.0,2024-03-15,4.5
1,E102,SARAH CONNOR,Cyber-Houston,"$95,000",34.0,05/20/2023,4.9
2,E102,SARAH CONNOR,Cyber-Houston,"$95,000",34.0,05/20/2023,4.9
3,E103,mike_tyson,AI_Ops-Dallas,N/A,NaN,2025/01/10,NaN
4,E104,Angela Camacho,IT-Austin,68000,29.0,invalid_date,4.2
5,E105,david_goggins,None,"$110,200",41.0,2022-11-01,5.0
6,E106,Elena Fisher,Cyber-Houston,"$-55,000",195.0,2024-07-22,3.8
7,E107,Marcus Fenix,AI_Ops-Houston,None,NaN,2026-02-14,NaN



Missing values count:
 Emp_ID               0
Full Name            0
Dept_Region          1
Salary_USD           1
Age                  2
Hire_Date            0
Performance_Score    2
dtype: int64


In [9]:
df = df_raw.copy()

# 1. Clean column headers (strip spaces, make lowercase)
df.columns = df.columns.str.strip().str.lower()

# 2. Drop duplicate rows based on Employee ID
df = df.drop_duplicates(subset=["emp_id"], keep="first").copy()

# 3. Clean names (replace underscores, strip spaces, Title Case)
df["full name"] = (
    df["full name"]
    .str.replace("_", " ", regex=False)
    .str.strip()
    .str.title()
)

# 4. Split 'dept_region' into separate 'department' and 'city' columns
df[["department", "city"]] = df["dept_region"].str.split("-", expand=True)
df["department"] = df["department"].str.replace("_", " ", regex=False)
df = df.drop(columns=["dept_region"])

# 5. Clean Salary (remove $ and commas, turn "N/A" to NaN, fix negative numbers)
df["salary_usd"] = (
    df["salary_usd"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["salary_usd"] = pd.to_numeric(df["salary_usd"], errors="coerce").abs()

# 6. Replace impossible ages (under 18 or over 80, like 195) with NaN
df.loc[(df["age"] < 18) | (df["age"] > 80), "age"] = np.nan

# 7. Standardize mixed date formats and turn "invalid_date" into NaT
df["hire_date"] = pd.to_datetime(df["hire_date"], format="mixed", errors="coerce")

print("=== STEP 2: FORMATTING FIXED & OUTLIERS CONVERTED TO NaN ===")
display(df)

=== STEP 2: FORMATTING FIXED & OUTLIERS CONVERTED TO NaN ===


,emp_id,full name,salary_usd,age,hire_date,performance_score,department,city
0,E101,John Smith,72500.0,28.0,2024-03-15,4.5,IT,Houston
1,E102,Sarah Connor,95000.0,34.0,2023-05-20,4.9,Cyber,Houston
3,E103,Mike Tyson,NaN,NaN,2025-01-10,NaN,AI Ops,Dallas
4,E104,Angela Camacho,68000.0,29.0,NaT,4.2,IT,Austin
5,E105,David Goggins,110200.0,41.0,2022-11-01,5.0,None,None
6,E106,Elena Fisher,55000.0,NaN,2024-07-22,3.8,Cyber,Houston
7,E107,Marcus Fenix,NaN,NaN,2026-02-14,NaN,AI Ops,Houston


In [10]:
numeric_cols = ["salary_usd", "age", "performance_score"]
categorical_cols = ["department", "city"]

# Fill missing numbers with the median (so outliers don't skew it)
num_imputer = SimpleImputer(strategy="median")
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# Fill missing text categories with the most frequent value
cat_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Fill missing hire dates with the median date
median_date = df["hire_date"].dropna().median()
df["hire_date"] = df["hire_date"].fillna(median_date)

# Save final clean dataset
df.to_csv("cleaned_ready_data.csv", index=False)

print("=== FINAL CLEANED DATASET (0 MISSING VALUES) ===")
display(df)
print("\nMissing values after cleaning:\n", df.isnull().sum())

=== FINAL CLEANED DATASET (0 MISSING VALUES) ===


,emp_id,full name,salary_usd,age,hire_date,performance_score,department,city
0,E101,John Smith,72500.0,28.0,2024-03-15 00:00:00,4.5,IT,Houston
1,E102,Sarah Connor,95000.0,34.0,2023-05-20 00:00:00,4.9,Cyber,Houston
3,E103,Mike Tyson,72500.0,31.5,2025-01-10 00:00:00,4.5,AI Ops,Dallas
4,E104,Angela Camacho,68000.0,29.0,2024-05-18 12:00:00,4.2,IT,Austin
5,E105,David Goggins,110200.0,41.0,2022-11-01 00:00:00,5.0,None,None
6,E106,Elena Fisher,55000.0,31.5,2024-07-22 00:00:00,3.8,Cyber,Houston
7,E107,Marcus Fenix,72500.0,31.5,2026-02-14 00:00:00,4.5,AI Ops,Houston



Missing values after cleaning:
 emp_id               0
full name            0
salary_usd           0
age                  0
hire_date            0
performance_score    0
department           1
city                 1
dtype: int64


In [11]:
# Let's fix the missing values in 'department' and 'city' explicitly using pandas fillna
# We fill missing values with the mode (most frequent value) of each column
for col in categorical_cols:
    most_frequent = df[col].mode()[0]
    df[col] = df[col].fillna(most_frequent)

# Save the corrected final clean dataset
df.to_csv("cleaned_ready_data.csv", index=False)

print("=== RE-EVALUATING MISSING VALUES ===")
display(df)
print("\nMissing values after fix:\n", df.isnull().sum())

=== RE-EVALUATING MISSING VALUES ===


,emp_id,full name,salary_usd,age,hire_date,performance_score,department,city
0,E101,John Smith,72500.0,28.0,2024-03-15 00:00:00,4.5,IT,Houston
1,E102,Sarah Connor,95000.0,34.0,2023-05-20 00:00:00,4.9,Cyber,Houston
3,E103,Mike Tyson,72500.0,31.5,2025-01-10 00:00:00,4.5,AI Ops,Dallas
4,E104,Angela Camacho,68000.0,29.0,2024-05-18 12:00:00,4.2,IT,Austin
5,E105,David Goggins,110200.0,41.0,2022-11-01 00:00:00,5.0,AI Ops,Houston
6,E106,Elena Fisher,55000.0,31.5,2024-07-22 00:00:00,3.8,Cyber,Houston
7,E107,Marcus Fenix,72500.0,31.5,2026-02-14 00:00:00,4.5,AI Ops,Houston



Missing values after fix:
 emp_id               0
full name            0
salary_usd           0
age                  0
hire_date            0
performance_score    0
department           0
city                 0
dtype: int64
